# Refresh Staging from Development

1. Clear old `Files/Staging/{Bronze,Silver,Gold}` data
2. Copy updated `Files/Development/...` into Staging
3. Refresh `Staging_Gold` **tables** from current Gold (keep views)



In [ ]:
from notebookutils import mssparkutils
from pyspark.sql import functions as F

WS = "718e8176-5d40-4a9c-88ff-50ac97ac49ba"
LH = "981fbe98-2f01-41d8-bf2f-a85e5cd9e2a2"
BASE = f"abfss://{WS}@onelake.dfs.fabric.microsoft.com/{LH}"

DEV = f"{BASE}/Files/Development"
STG = f"{BASE}/Files/Staging"

def exists(path):
    try:
        return mssparkutils.fs.exists(path)
    except Exception:
        return False

def rm(path):
    if exists(path):
        mssparkutils.fs.rm(path, True)
        print(f"[RM] {path}")
    else:
        print(f"[SKIP RM] missing {path}")

def cp(src, dst):
    if not exists(src):
        print(f"[SKIP CP] missing src {src}")
        return
    # ensure parent
    parent = "/".join(dst.rstrip("/").split("/")[:-1])
    if parent and not exists(parent):
        mssparkutils.fs.mkdirs(parent)
    if exists(dst):
        mssparkutils.fs.rm(dst, True)
    mssparkutils.fs.cp(src, dst, True)
    print(f"[CP] {src} -> {dst}")

print("helpers ready")



In [ ]:
# ---- 1) Clear old Staging layer folders ----
for layer in ["Bronze", "Silver", "Gold"]:
    rm(f"{STG}/{layer}")
    mssparkutils.fs.mkdirs(f"{STG}/{layer}")
    print(f"[OK] cleared Staging/{layer}")



In [ ]:
# ---- 2) Copy Development -> Staging (updated data) ----

# Bronze: Meta + Google from Development
cp(f"{DEV}/Bronze/Meta_ads", f"{STG}/Bronze/Meta_ads")
cp(f"{DEV}/Bronze/Google_ads", f"{STG}/Bronze/Google_ads")

# Silver: Meta + Google from Development
cp(f"{DEV}/Silver/meta_ads", f"{STG}/Silver/meta_ads")
cp(f"{DEV}/Silver/GoogleAds", f"{STG}/Silver/GoogleAds")
# also copy Meta notebook folder if present
if exists(f"{DEV}/Silver/Meta"):
    cp(f"{DEV}/Silver/Meta", f"{STG}/Silver/Meta")

# Gold files: Development gold outputs + unified
cp(f"{DEV}/Gold/GoogleAds", f"{STG}/Gold/GoogleAds")
if exists(f"{DEV}/Gold/Meta"):
    cp(f"{DEV}/Gold/Meta", f"{STG}/Gold/Meta")
if exists(f"{DEV}/Gold/rpt_unified_ad_performance"):
    cp(f"{DEV}/Gold/rpt_unified_ad_performance", f"{STG}/Gold/rpt_unified_ad_performance")
if exists(f"{DEV}/Gold/exports"):
    cp(f"{DEV}/Gold/exports", f"{STG}/Gold/exports")

# Also sync production Files/Gold meta rpt if present (latest enriched)
if exists(f"{BASE}/Files/Gold/rpt_meta_ad_performance_daily"):
    cp(f"{BASE}/Files/Gold/rpt_meta_ad_performance_daily", f"{STG}/Gold/rpt_meta_ad_performance_daily")

print("[OK] Staging files replaced from Development")



In [ ]:
# ---- 3) Staging_Gold managed TABLES only (exclude views) ----
spark.sql("CREATE SCHEMA IF NOT EXISTS Staging_Gold")
spark.sql("CREATE SCHEMA IF NOT EXISTS Gold")

# Drop old non-view tables in Staging_Gold, then rewrite from Gold
old_tables = [
    "dim_account", "dim_ad", "dim_adset", "dim_campaign", "dim_date",
    "fact_ad_performance_daily",
]
for t in old_tables:
    spark.sql(f"DROP TABLE IF EXISTS Staging_Gold.{t}")
    print(f"[DROP] Staging_Gold.{t}")

# Prefer current Gold reporting tables as Staging_Gold content
# Copy rpt tables into Staging_Gold (tables, not views)
for t in [
    "rpt_meta_ad_performance_daily",
    "rpt_google_ad_performance_daily",
    "rpt_unified_ad_performance",
]:
    df = spark.table(f"Gold.{t}")
    (
        df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable(f"Staging_Gold.{t}")
    )
    print(f"[OK] Staging_Gold.{t}: {spark.table(f'Staging_Gold.{t}').count():,}")

# Also refresh dims/fact in Staging_Gold from Gold if they exist
for t in ["dim_account", "dim_ad", "dim_adset", "dim_campaign", "dim_date", "fact_ad_performance_daily"]:
    try:
        df = spark.table(f"Gold.{t}")
        (
            df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
            .saveAsTable(f"Staging_Gold.{t}")
        )
        print(f"[OK] Staging_Gold.{t}: {spark.table(f'Staging_Gold.{t}').count():,}")
    except Exception as e:
        print(f"[SKIP] Gold.{t}: {e}")

# Also write file copies under Files/Staging/Gold for the rpt tables
for t in ["rpt_meta_ad_performance_daily", "rpt_google_ad_performance_daily", "rpt_unified_ad_performance"]:
    df = spark.table(f"Staging_Gold.{t}")
    path = f"{STG}/Gold/{t}"
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(path)
    print(f"[OK] files {path}")

print("[OK] Staging_Gold tables refreshed (views untouched)")



In [ ]:
# ---- 4) Recreate Staging_Gold views (still present; point at Staging_Gold tables where possible) ----
# Keep views; prefer Staging_Gold.rpt_* if available else Gold.rpt_*

spark.sql('''
CREATE OR REPLACE VIEW Staging_Gold.vw_ad_performance AS
SELECT * FROM Staging_Gold.rpt_unified_ad_performance
''')
spark.sql('''
CREATE OR REPLACE VIEW Staging_Gold.vw_unified_ad_performance AS
SELECT * FROM Staging_Gold.rpt_unified_ad_performance
''')
spark.sql('''
CREATE OR REPLACE VIEW Staging_Gold.vw_meta_ad_performance AS
SELECT * FROM Staging_Gold.rpt_meta_ad_performance_daily
''')
spark.sql('''
CREATE OR REPLACE VIEW Staging_Gold.vw_google_ad_performance AS
SELECT * FROM Staging_Gold.rpt_google_ad_performance_daily
''')
spark.sql('''
CREATE OR REPLACE VIEW Staging_Gold.vw_adset_performance AS
SELECT
  platform, full_date, year, month, month_name, day_name,
  account_id, MAX(account_name) AS account_name,
  campaign_id, MAX(campaign_name) AS campaign_name,
  adset_id, MAX(adset_name) AS adset_name, MAX(adset_status) AS adset_status,
  MAX(optimization_goal) AS optimization_goal,
  MAX(age_range) AS age_range, MAX(geo_cities) AS geo_cities, MAX(geo_regions) AS geo_regions,
  SUM(impressions) AS impressions, SUM(reach) AS reach, SUM(clicks) AS clicks,
  SUM(spend) AS spend, SUM(leads) AS leads,
  CASE WHEN SUM(clicks) > 0 THEN SUM(spend)/SUM(clicks) ELSE NULL END AS cpc,
  CASE WHEN SUM(impressions) > 0 THEN (SUM(spend)/SUM(impressions))*1000 ELSE NULL END AS cpm,
  CASE WHEN SUM(leads) > 0 THEN SUM(spend)/SUM(leads) ELSE NULL END AS cost_per_lead,
  COUNT(DISTINCT ad_id) AS ad_count
FROM Staging_Gold.rpt_unified_ad_performance
GROUP BY platform, full_date, year, month, month_name, day_name, account_id, campaign_id, adset_id
''')
spark.sql('''
CREATE OR REPLACE VIEW Staging_Gold.vw_campaign_performance AS
SELECT
  platform, full_date, year, month, month_name, day_name,
  account_id, MAX(account_name) AS account_name,
  campaign_id, MAX(campaign_name) AS campaign_name,
  MAX(campaign_status) AS campaign_status,
  MAX(campaign_channel_or_objective) AS campaign_channel_or_objective,
  MAX(daily_budget_inr) AS daily_budget_inr,
  SUM(impressions) AS impressions, SUM(reach) AS reach, SUM(clicks) AS clicks,
  SUM(spend) AS spend, SUM(leads) AS leads,
  CASE WHEN SUM(clicks) > 0 THEN SUM(spend)/SUM(clicks) ELSE NULL END AS cpc,
  CASE WHEN SUM(impressions) > 0 THEN (SUM(spend)/SUM(impressions))*1000 ELSE NULL END AS cpm,
  CASE WHEN SUM(leads) > 0 THEN SUM(spend)/SUM(leads) ELSE NULL END AS cost_per_lead,
  COUNT(DISTINCT adset_id) AS adset_count, COUNT(DISTINCT ad_id) AS ad_count
FROM Staging_Gold.rpt_unified_ad_performance
GROUP BY platform, full_date, year, month, month_name, day_name, account_id, campaign_id
''')
print("[OK] Staging_Gold views reattached to Staging_Gold tables")



In [ ]:
# ---- Validate ----
print("\nStaging folders:")
for p in [
    f"{STG}/Bronze/Meta_ads", f"{STG}/Bronze/Google_ads",
    f"{STG}/Silver/meta_ads", f"{STG}/Silver/GoogleAds",
    f"{STG}/Gold/GoogleAds", f"{STG}/Gold/rpt_unified_ad_performance",
    f"{STG}/Gold/rpt_meta_ad_performance_daily",
]:
    print(" ", p, "EXISTS" if exists(p) else "MISSING")

print("\nStaging_Gold objects:")
spark.sql("SHOW TABLES IN Staging_Gold").show(50, truncate=False)
spark.sql("SHOW VIEWS IN Staging_Gold").show(50, truncate=False)

for t in ["rpt_unified_ad_performance", "rpt_meta_ad_performance_daily", "rpt_google_ad_performance_daily"]:
    print(t, spark.table(f"Staging_Gold.{t}").count())
print("STAGING_REFRESH_FROM_DEVELOPMENT_COMPLETE")

